In [1]:
import os
import numpy as np
import pandas as pd
import random
import json
from itertools import product
from datetime import datetime
import traceback

# ======================================================================
# CONFIGURATION PARAMETERS
# ======================================================================

CONFIG = {
    'repetitions': 12,            # how many times each condition × SOA is repeated
    'num_participants': 5,       # default number of participants
    'included_conditions': {
        'inhalation': True,
        'exhalation': True,
        'baseline': True,
    },
    'catch_trial_percentage': 10,
    'soa_conditions_ms': [190, 400, 700, 1000, 1500],
    'jitter_options_ms': [100, 200, 300, 400, 500],
    'debug_mode': True,
    # CSV file with all the breath-hold timestamps
    'timestamps_csv': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv",
    'paths': {
        'experiment_log_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
    }
}

# Noise types for looming (these can be any labels you want)
LOOMING_STIMULI = ['blue', 'brown', 'pink', 'white']

# The boundaries of the segment we will replicate if needed
SEGMENT_START = "02:08.4"  # 13th hold ends
SEGMENT_END   = "04:56.4"  # 34th hold ends

# ======================================================================
# CLASS: PPSDesignGenerator
# ======================================================================

class PPSDesignGenerator:
    """Generates CSV design files for Peripersonal Space (PPS) experiments (no audio)."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Prepare output directory
        self.experiment_log_dir = self.config['paths']['experiment_log_dir']
        os.makedirs(self.experiment_log_dir, exist_ok=True)
        
        # Calculate trial counts
        self.trial_counts = self._calculate_trial_counts()
        self.conditions = [cond for cond, inc in self.config['included_conditions'].items() if inc]
        
        # Internal property for the timestamps
        self.timestamps_df = None
        
        if self.config['debug_mode']:
            print("Initialized PPSDesignGenerator with:")
            print(f"  - Repetitions: {self.config['repetitions']}")
            print(f"  - Conditions: {self.conditions}")
            print(f"  - SOA values: {self.config['soa_conditions_ms']}")
            print(f"  - Total trials per participant: {sum(self.trial_counts.values())}")
            print(f"  - Output directory: {self.experiment_log_dir}")
    
    def _calculate_trial_counts(self):
        """Calculate how many trials for each condition (inhale/exhale/baseline) plus catch."""
        reps = self.config['repetitions']
        soa_count = len(self.config['soa_conditions_ms'])
        stimulus_types = LOOMING_STIMULI
        catch_percentage = self.config['catch_trial_percentage']
        
        trial_counts = {}
        for condition, include in self.config['included_conditions'].items():
            if include:
                # (# noise types) × (# SOA) × (reps)
                trial_counts[condition] = len(stimulus_types) * soa_count * reps
        
        regular_trials = sum(trial_counts.values())
        catch_trials = int(np.ceil(regular_trials * catch_percentage / 100))
        trial_counts['catch'] = catch_trials
        
        if self.config['debug_mode']:
            print("Trial count calculation:")
            for cond, count in trial_counts.items():
                print(f"  {cond}: {count} trials")
            print(f"  Total: {sum(trial_counts.values())} trials")
        
        return trial_counts
    
    def _expand_timestamps_if_needed(self):
        """
        Expand self.timestamps_df if we don't have enough rows to cover all trials.
        We replicate the segment from SEGMENT_START to SEGMENT_END, offsetting times
        and labeling each repeated block with repetition=1,2,3,... The original rows
        get repetition=0.
        """
        if self.timestamps_df is None or self.timestamps_df.empty:
            return
        
        # 1) Figure out how many total trials we have
        #    (each trial needs exactly one breath hold).
        total_trials = sum(self.trial_counts.values())
        
        # 2) Check how many breath-hold rows are in the original CSV
        original_count = len(self.timestamps_df)
        if original_count >= total_trials:
            if self.config['debug_mode']:
                print(f"No expansion needed; CSV has {original_count} holds, need {total_trials}.")
            # Just set repetition=0 for all
            self.timestamps_df["repetition"] = 0
            return
        
        if self.config['debug_mode']:
            print(f"Expanding timestamps: have {original_count}, need {total_trials}.")
        
        # Sort the DataFrame by milliseconds to ensure order
        self.timestamps_df = self.timestamps_df.sort_values(by="milliseconds").reset_index(drop=True)
        
        # Identify the row indices for the segment we replicate
        seg_start_idx = self.timestamps_df.index[self.timestamps_df["timestamp"] == SEGMENT_START].tolist()
        seg_end_idx   = self.timestamps_df.index[self.timestamps_df["timestamp"] == SEGMENT_END].tolist()
        
        if not seg_start_idx or not seg_end_idx:
            raise ValueError(f"Could not find start='{SEGMENT_START}' or end='{SEGMENT_END}' in CSV.")
        
        start_i = seg_start_idx[0]
        end_i   = seg_end_idx[0]
        if end_i <= start_i:
            raise ValueError(f"Segment end index <= start index. Check your timestamps.")
        
        intro_df  = self.timestamps_df.iloc[:start_i].copy()
        repeat_df = self.timestamps_df.iloc[start_i:end_i+1].copy()
        outro_df  = self.timestamps_df.iloc[end_i+1:].copy()
        
        # Label original rows with repetition=0
        intro_df["repetition"]  = 0
        repeat_df["repetition"] = 0
        outro_df["repetition"]  = 0
        
        # Check how many we have if we keep the entire original CSV
        current_count = len(intro_df) + len(repeat_df) + len(outro_df)
        seg_len = len(repeat_df)
        
        if current_count >= total_trials:
            # We can just keep them as-is
            self.timestamps_df = pd.concat([intro_df, repeat_df, outro_df], ignore_index=True)
            return
        
        # Otherwise, replicate the repeat block until we have enough
        block_duration_ms = repeat_df["milliseconds"].iloc[-1] - repeat_df["milliseconds"].iloc[0]
        block_duration_samples = repeat_df["sample_index"].iloc[-1] - repeat_df["sample_index"].iloc[0]
        
        # For shifting the "timestamp" column (MM:SS.s)
        block_duration_sec = block_duration_ms / 1000.0
        
        def shift_timestamp(ts, offset_sec):
            if not isinstance(ts, str):
                return ts
            parts = ts.split(":")
            if len(parts) != 2:
                return ts
            mm = int(parts[0])
            ss = float(parts[1])
            total_s = mm*60 + ss
            new_s = total_s + offset_sec
            new_mm = int(new_s // 60)
            new_ss = new_s % 60
            return f"{new_mm:02}:{new_ss:04.1f}"
        
        repeated_blocks = []
        offset_sec   = block_duration_sec
        offset_ms    = block_duration_ms
        offset_samps = block_duration_samples
        
        repetition_index = 1  # original = 0, so first copy = 1
        
        while current_count < total_trials:
            block_copy = repeat_df.copy()
            
            # shift each row's timestamps
            block_copy["timestamp"] = block_copy["timestamp"].apply(lambda t: shift_timestamp(t, offset_sec))
            block_copy["milliseconds"] = block_copy["milliseconds"] + offset_ms
            block_copy["sample_index"] = block_copy["sample_index"] + offset_samps
            
            block_copy["repetition"] = repetition_index
            
            repeated_blocks.append(block_copy)
            
            # increment
            current_count += seg_len
            repetition_index += 1
            offset_sec   += block_duration_sec
            offset_ms    += block_duration_ms
            offset_samps += block_duration_samples
        
        # Combine everything
        self.timestamps_df = pd.concat([intro_df, repeat_df] + repeated_blocks + [outro_df], ignore_index=True)
        if self.config['debug_mode']:
            print(f"Final expanded CSV has {len(self.timestamps_df)} breath holds (needed {total_trials}).")
    
    def parse_timestamps(self):
        """Parse the box breathing timestamps CSV once, storing in self.timestamps_df, then expand if needed."""
        if self.timestamps_df is not None:
            return self.timestamps_df
        
        csv_path = self.config['timestamps_csv']
        if not os.path.exists(csv_path):
            print(f"ERROR: Timestamps CSV not found at {csv_path}")
            self.timestamps_df = pd.DataFrame()
            return self.timestamps_df
        
        try:
            df = pd.read_csv(csv_path)
            # Ensure 'breathphase' column is present
            if 'breathphase' not in df.columns:
                df['breathphase'] = ['inhale' if i % 2 == 0 else 'exhale' for i in range(len(df))]
            # Add time_seconds for convenience
            if 'time_seconds' not in df.columns:
                df['time_seconds'] = df['milliseconds'] / 1000.0
            
            # The original CSV, no repetition assigned yet
            df["repetition"] = 0
            
            self.timestamps_df = df
            
            if self.config['debug_mode']:
                print(f"Parsed {len(df)} timestamps from {csv_path}")
            
            # Now expand if needed
            self._expand_timestamps_if_needed()
            
            return self.timestamps_df
        except Exception as e:
            print(f"Error parsing timestamps CSV: {e}")
            traceback.print_exc()
            self.timestamps_df = pd.DataFrame()
            return self.timestamps_df
    
    def generate_counterbalanced_design(self, participant_id):
        """
        Generate a DataFrame with all trials (including baseline, catch, etc.)
        with random jitter, SOA, noise type, etc.
        """
        soa_conditions = self.config['soa_conditions_ms']
        jitter_options = self.config['jitter_options_ms']
        stimulus_types = LOOMING_STIMULI
        
        all_trials = []
        
        # Build the main conditions (inhale, exhale, baseline)
        for condition, count in self.trial_counts.items():
            if condition == 'catch' or count == 0:
                continue
            
            # All combos of (noise type × SOA), repeated 'reps' times
            stim_soa_combinations = list(product(stimulus_types, soa_conditions))
            reps_needed = self.config['repetitions']
            
            full_combinations = stim_soa_combinations * reps_needed
            random.shuffle(full_combinations)
            
            for stim_type, soa in full_combinations:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': condition,  # inhalation, exhalation, baseline
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': True  # baseline & main trials have tactile
                })
        
        # Add catch trials (looming only, no tactile)
        catch_count = self.trial_counts.get('catch', 0)
        if catch_count > 0:
            catch_combos = list(product(stimulus_types, soa_conditions))
            while len(catch_combos) < catch_count:
                catch_combos.extend(catch_combos)
            catch_sample = random.sample(catch_combos, catch_count)
            
            for stim_type, soa in catch_sample:
                all_trials.append({
                    'participant_id': participant_id,
                    'trial_type': 'catch',
                    'stimulus_type': stim_type,
                    'soa_value_ms': soa,
                    'jitter_ms': random.choice(jitter_options),
                    'is_tactile': False
                })
        
        # Convert to DataFrame and randomize order
        design_df = pd.DataFrame(all_trials)
        design_df = design_df.sample(frac=1, random_state=participant_id*100).reset_index(drop=True)
        
        # Add a trial_number
        design_df['trial_number'] = range(1, len(design_df) + 1)
        
        return design_df
    
    def assign_breath_holds(self, design_df):
        """
        Assign each trial to an inhale or exhale row in self.timestamps_df.
        Also compute derived timestamps in 'MM:SS.s' format for retention,
        looming, and tactile. 
        """
        timestamps_df = self.parse_timestamps()  # ensures expansion is done
        if timestamps_df.empty:
            # If we have no timestamps, just return the design with no assignment
            design_df['breathphase'] = None
            design_df['milliseconds'] = None
            design_df['jittered_ms'] = None
            design_df['soa_ms'] = None
            design_df['timestamp_original'] = None
            design_df['timestamp_after_jitter'] = None
            design_df['timestamp_with_soa'] = None
            return design_df
        
        # We'll assume the first ~10% are "intro" and skip them for assignment
        skip_count = int(len(timestamps_df) * 0.1)
        usable = timestamps_df.iloc[skip_count:].copy()
        
        # Separate inhale vs exhale
        inhale_df = usable[usable['breathphase'] == 'inhale'].index.tolist()
        exhale_df = usable[usable['breathphase'] == 'exhale'].index.tolist()
        
        # For each condition, we attempt to split half the trials to inhale, half to exhale
        trial_assignments = {}
        
        # Group by condition
        for condition in design_df['trial_type'].unique():
            condition_mask = (design_df['trial_type'] == condition)
            condition_indices = design_df[condition_mask].index
            condition_count = len(condition_indices)
            
            half_inhale = condition_count // 2
            half_exhale = condition_count - half_inhale
            
            used_inhales = inhale_df[:half_inhale]
            used_exhales = exhale_df[:half_exhale]
            
            # Actually map them onto the design indices
            for d_idx, t_idx in zip(condition_indices[:half_inhale], used_inhales):
                row_ts = timestamps_df.loc[t_idx]
                trial_assignments[d_idx] = row_ts
            
            for d_idx, t_idx in zip(condition_indices[half_inhale:half_inhale+half_exhale], used_exhales):
                row_ts = timestamps_df.loc[t_idx]
                trial_assignments[d_idx] = row_ts
            
            # Remove those from the "pool"
            inhale_df = inhale_df[half_inhale:]
            exhale_df = exhale_df[half_exhale:]
        
        # Now fill in columns
        design_df['breathphase'] = design_df.index.map(
            lambda i: trial_assignments[i]['breathphase'] if i in trial_assignments else None
        )
        design_df['milliseconds'] = design_df.index.map(
            lambda i: trial_assignments[i]['milliseconds'] if i in trial_assignments else None
        )
        
        # Jittered_ms = base + jitter
        design_df['jittered_ms'] = design_df.apply(
            lambda row: row['milliseconds'] + row['jitter_ms'] if pd.notnull(row['milliseconds']) else None,
            axis=1
        )
        
        # If not a catch trial, tactile is offset from looming by the SOA
        design_df['soa_ms'] = design_df.apply(
            lambda row: row['jittered_ms'] + row['soa_value_ms']
            if (pd.notnull(row['jittered_ms']) and row['trial_type'] != 'catch') else None,
            axis=1
        )
        
        # Convert ms → "MM:SS.s"
        def ms_to_timestamp(ms):
            if pd.isnull(ms):
                return None
            m = int(ms // 60000)
            s = (ms % 60000) / 1000.0
            return f"{m:02}:{s:.1f}"
        
        design_df['timestamp_original'] = design_df['milliseconds'].apply(ms_to_timestamp)
        design_df['timestamp_after_jitter'] = design_df['jittered_ms'].apply(ms_to_timestamp)
        design_df['timestamp_with_soa'] = design_df['soa_ms'].apply(ms_to_timestamp)
        
        return design_df
    
    def finalize_design_csv(self, design_df):
        """
        1) Rename columns to final naming:
           - stimulus_type -> looming_stimulus_type
           - timestamp_original -> retentionphase_timestamp
           - timestamp_after_jitter -> looming_stimulus_timestamp
           - timestamp_with_soa -> tactile_stimulus_timestamp
        2) For baseline trials: set looming_stimulus_type='none', looming_stimulus_timestamp=NaN
        3) For catch trials: set tactile_stimulus_timestamp=NaN
        4) Return the final DataFrame (ready to save).
        """
        # Rename stimulus_type -> looming_stimulus_type
        design_df.rename(columns={'stimulus_type': 'looming_stimulus_type'},
                         inplace=True)
        
        # baseline => no looming
        baseline_mask = (design_df['trial_type'] == 'baseline')
        design_df.loc[baseline_mask, 'looming_stimulus_type'] = 'none'
        
        # Rename these columns
        design_df.rename(columns={
            'timestamp_original':      'retentionphase_timestamp',
            'timestamp_after_jitter':  'looming_stimulus_timestamp',
            'timestamp_with_soa':      'tactile_stimulus_timestamp'
        }, inplace=True)
        
        # baseline => looming_stimulus_timestamp=NaN
        design_df.loc[baseline_mask, 'looming_stimulus_timestamp'] = np.nan
        
        # catch => no tactile => tactile_stimulus_timestamp=NaN
        catch_mask = (design_df['trial_type'] == 'catch')
        design_df.loc[catch_mask, 'tactile_stimulus_timestamp'] = np.nan
        
        # Optionally reorder columns
        final_cols = [
            'participant_id',
            'trial_number',
            'trial_type',             # inhalation, exhalation, baseline, catch
            'breathphase',            # inhale or exhale
            'retentionphase_timestamp',
            'looming_stimulus_timestamp',
            'tactile_stimulus_timestamp',
            'looming_stimulus_type',
            'soa_value_ms',
            'jitter_ms',
            'is_tactile'
        ]
        # Keep only columns that actually exist in the DF
        final_cols = [c for c in final_cols if c in design_df.columns]
        design_df = design_df[final_cols]
        
        return design_df
    
    def process_participant(self, participant_id):
        """
        Generate the design, assign breath holds, finalize columns,
        and save as a CSV for one participant.
        """
        try:
            if self.config['debug_mode']:
                print(f"\n=== Processing participant {participant_id} ===")
            
            # 1) Generate base design (counterbalanced combos)
            design_df = self.generate_counterbalanced_design(participant_id)
            
            # 2) Assign each trial to a breath hold (inhale or exhale)
            design_df = self.assign_breath_holds(design_df)
            
            # 3) Finalize columns (rename, set baseline/catch to NA, etc.)
            design_df = self.finalize_design_csv(design_df)
            
            # 4) Save CSV
            out_path = os.path.join(self.experiment_log_dir,
                                    f"participant_{participant_id}_design.csv")
            design_df.to_csv(out_path, index=False)
            
            if self.config['debug_mode']:
                print(f"Saved participant {participant_id} design to {out_path}")
                print(design_df.head())
            
            return True, out_path
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None, num_participants=None):
        """
        Generate the design CSVs for multiple participants.
        """
        if participant_ids is None and num_participants is None:
            num_participants = self.config.get('num_participants', 5)
        
        if participant_ids is None:
            participant_ids = list(range(num_participants))
        
        if self.config['debug_mode']:
            print("=== Starting PPSDesignGenerator ===")
            print(f"Repetitions: {self.config['repetitions']}")
            print(f"Participants: {participant_ids}")
        
        results = {}
        for pid in participant_ids:
            success, path = self.process_participant(pid)
            results[pid] = {
                'success': success,
                'csv_path': path
            }
        
        return results

# ======================================================================
# MAIN
# ======================================================================

def main():
    generator = PPSDesignGenerator(CONFIG)
    results = generator.run()  # by default, uses num_participants=5
    
    # Print summary
    print("\n=== Generation Summary ===")
    for pid, info in results.items():
        if info['success']:
            print(f"Participant {pid}: CSV generated at {info['csv_path']}")
        else:
            print(f"Participant {pid}: ERROR - no CSV generated.")

if __name__ == "__main__":
    main()


Trial count calculation:
  inhalation: 240 trials
  exhalation: 240 trials
  baseline: 240 trials
  catch: 72 trials
  Total: 792 trials
Initialized PPSDesignGenerator with:
  - Repetitions: 12
  - Conditions: ['inhalation', 'exhalation', 'baseline']
  - SOA values: [190, 400, 700, 1000, 1500]
  - Total trials per participant: 792
  - Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
=== Starting PPSDesignGenerator ===
Repetitions: 12
Participants: [0, 1, 2, 3, 4]

=== Processing participant 0 ===
Parsed 38 timestamps from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_timestamps.csv
Expanding timestamps: have 38, need 792.
Final expanded CSV has 808 breath holds (needed 792).
Error processing participant 0: [Errno 13] Permission denied: 'C:\\Users\\cogpsy-vrlab\\Documents\\GitHub\\BreathingSpace\\Level1_AudioGeneration\\ExperimentLog\\participant_0_design.csv'

=== Processing participan

Traceback (most recent call last):
  File "C:\Users\cogpsy-vrlab\AppData\Local\Temp\ipykernel_18584\1894142644.py", line 466, in process_participant
    design_df.to_csv(out_path, index=False)
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\core\generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\formats\csvs.py", line 251, in save
    with get_handle(
         ^^^^^^^^^^^
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\pandas\io\common.py", line 873, in get_handle
    handle = open(
             ^^^^^
PermissionE

Saved participant 3 design to C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog\participant_3_design.csv
   participant_id  trial_number  trial_type breathphase  \
0               3             1    baseline      inhale   
1               3             2    baseline      inhale   
2               3             3  exhalation      inhale   
3               3             4  inhalation      inhale   
4               3             5    baseline      inhale   

  retentionphase_timestamp looming_stimulus_timestamp  \
0                  10:48.4                        NaN   
1                   11:4.4                        NaN   
2                  41:20.4                    41:20.5   
3                   72:0.4                     72:0.6   
4                  11:20.4                        NaN   

  tactile_stimulus_timestamp looming_stimulus_type  soa_value_ms  jitter_ms  \
0                    10:49.2                  none           700        100  